# آزمون واقعی قسمت سوم

این دفترچه به اینترنت و دسترسی به TSETMC نیاز دارد. نماد مبهم به‌صورت حدسی انتخاب نمی‌شود.

## سلول ۱ — ورود کتابخانه‌ها

In [ ]:
from pathlib import Path
import pandas as pd
import pydara
from pydara import tse
print(pydara.__version__)

## سلول ۲ — تعریف توابع

In [ ]:
def show_report(result):
    display(pd.DataFrame([result.summary]))
    display(result.data[["date", "symbol", "closing_price", "volume"]].tail())

## سلول ۳ — تنظیمات

In [ ]:
SYMBOL = 'فولاد'
OUTPUT = Path.cwd() / "dara_live_exports"
# Use a verified ins_code rather than guessing among ambiguous symbols.
# SYMBOL = tse.InstrumentRef.by_ins_code("46348559193224090")

## سلول ۴ — اجرا

In [ ]:
with tse.Client() as client:
    result = client.sync_history(SYMBOL, timeout=30, on_storage_error="raise")
    stored = client.load_stored_history(SYMBOL, include_jdate=True)
    status = client.history_status()
    receipt = tse.save(stored, OUTPUT / "history.csv", overwrite=True)

## سلول ۵ — گزارش نتیجه

In [ ]:
show_report(result)
display(status)
print("Saved:", receipt.destination)
restored = tse.load(OUTPUT / "history.csv")
pd.testing.assert_frame_equal(stored, restored)
print("CSV round trip passed.")